# DoRA с нуля на PyTorch

Реализуем адаптер для `nn.Linear` по формуле (5) статьи DoRA и используем вариант обратного прохода из §4.3. Затем предобучим маленькую сеть, адаптируем её к новой задаче и объединим веса для инференса.

Для первых разделов нужны Python 3 и PyTorch. В Jupyter выберите ядро с установленным PyTorch и выполните **Run All**. Если пакет ещё не установлен, выполните `%pip install torch` в отдельной ячейке. Синтетическая проверка работает на CPU. В последнем разделе дообучим DistilBERT на AG News с тем же адаптером.

[DoRA: Weight-Decomposed Low-Rank Adaptation, arXiv v6](https://arxiv.org/html/2402.09353v6#S4)

In [8]:
import math

import torch
from torch import nn
from torch.nn import functional as F

## От формулы к `nn.Linear`

У каждой строки веса линейного слоя есть длина и направление. DoRA обучает отдельный параметр длины для каждого выходного признака, а изменение направления задаёт низкоранговым произведением $BA$.

$$
V = W_0 + BA, \qquad
W_{\mathrm{DoRA}}[i, :] = m_i \frac{V[i, :]}{\lVert V[i, :] \rVert_2}.
$$

Здесь $i$ — индекс выходного признака. В записи статьи одному выходному признаку соответствует столбец. В `nn.Linear` веса хранятся как `[out_features, in_features]`, поэтому считаем норму **по строкам**, сворачивая входную ось `dim=1`.

| Тензор | Размер | Роль |
| --- | --- | --- |
| $W_0$ | `[out_features, in_features]` | Замороженный вес исходного слоя |
| $A$ | `[rank, in_features]` | Обучаемая матрица с инициализацией Kaiming uniform |
| $B$ | `[out_features, rank]` | Обучаемая матрица, в начале заполненная нулями |
| $m$ | `[out_features, 1]` | Обучаемый столбец, в начале равный нормам строк $W_0$ |

На старте $BA=0$, а для каждой ненулевой строки $m_i=\lVert W_0[i,:]\rVert_2$, поэтому слой даёт тот же результат, что исходный. Смещение исходного слоя копируем и замораживаем.

В §4.2 статьи градиент проходит через норму. В §4.3 норму пересчитывают на каждом прямом проходе, но считают константой при обратном проходе. В коде ниже `detach()` применяется только при вычислении знаменателя. Числитель и параметр `magnitude` остаются в графе.

In [9]:
class DoRALinear(nn.Module):
    """DoRA для предобученного nn.Linear с весами float32 или float64."""

    def __init__(self, base_layer: nn.Linear, rank: int):
        super().__init__()
        self.in_features = base_layer.in_features
        self.out_features = base_layer.out_features
        if not 1 <= rank <= min(self.in_features, self.out_features):
            raise ValueError("rank должен быть от 1 до min(in_features, out_features)")

        # Буферы сохраняются в state_dict и перемещаются вместе с модулем.
        weight = base_layer.weight.detach().clone()
        bias = base_layer.bias
        self.register_buffer("base_weight", weight)
        self.register_buffer("base_bias", None if bias is None else bias.detach().clone())

        self.lora_a = nn.Parameter(weight.new_empty(rank, self.in_features))
        self.lora_b = nn.Parameter(weight.new_zeros(self.out_features, rank))
        self.magnitude = nn.Parameter(torch.linalg.vector_norm(weight, dim=1, keepdim=True))

        nn.init.kaiming_uniform_(self.lora_a, a=math.sqrt(5))

    def effective_weight(self) -> torch.Tensor:
        """Вычисляет W_DoRA по формуле (5) в ориентации nn.Linear."""
        direction = self.base_weight + self.lora_b @ self.lora_a
        row_norm = torch.linalg.vector_norm(direction.detach(), dim=1, keepdim=True)

        # У нулевой строки нет направления. Деление на 1 сохраняет нулевой результат.
        safe_norm = torch.where(row_norm == 0, 1.0, row_norm)
        return direction * (self.magnitude / safe_norm)

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        return F.linear(inputs, self.effective_weight(), self.base_bias)

    @torch.no_grad()
    def to_linear(self) -> nn.Linear:
        """Возвращает отдельный nn.Linear с объединёнными весами для инференса."""
        weight = self.effective_weight()
        merged = nn.Linear(
            self.in_features,
            self.out_features,
            bias=self.base_bias is not None,
            device=weight.device,
            dtype=weight.dtype,
        )
        merged.weight.copy_(weight)
        if merged.bias is not None:
            merged.bias.copy_(self.base_bias)
        return merged.requires_grad_(False).eval()

Здесь `effective_weight()` явно строит плотную матрицу, чтобы код повторял формулу. Такая реализация рассчитана на небольшие примеры. У полностью нулевой строки базового веса на старте одновременно нулевые направление и масштаб. В этом случае строка сохраняет нулевой вес и адаптация этой строки не стартует.

## Проверки слоя

Возьмём прямоугольный слой, чтобы ошибка с осью нормы стала видна. Проверим совпадение с базовым слоем, формулу после изменения адаптера и градиенты первого шага. Поскольку $B=0$ на старте, градиент $A$ в этот момент равен нулю.

In [10]:
torch.manual_seed(7)

base_layer = nn.Linear(5, 3)
adapter = DoRALinear(base_layer, rank=2)
inputs = torch.randn(2, 4, 5)  # Например, [batch, tokens, in_features].

with torch.no_grad():
    torch.testing.assert_close(adapter(inputs), base_layer(inputs))

# Для loss = sum(W_DoRA) при старте m / norm = 1.
# С остановкой градиента в знаменателе d(loss)/dB = ones_like(W0) @ A.T.
adapter.effective_weight().sum().backward()
expected_b_grad = torch.ones_like(adapter.base_weight) @ adapter.lora_a.detach().T

torch.testing.assert_close(adapter.lora_b.grad, expected_b_grad)
torch.testing.assert_close(adapter.lora_a.grad, torch.zeros_like(adapter.lora_a))
unit_rows = adapter.base_weight / adapter.magnitude.detach()
expected_magnitude_grad = unit_rows.sum(dim=1, keepdim=True)
torch.testing.assert_close(adapter.magnitude.grad, expected_magnitude_grad)
assert set(dict(adapter.named_parameters())) == {"lora_a", "lora_b", "magnitude"}
assert base_layer.weight.grad is None and base_layer.bias.grad is None

with torch.no_grad():
    adapter.lora_b.normal_(std=0.2)
    adapter.magnitude.mul_(1.5)
    direction = adapter.base_weight + adapter.lora_b @ adapter.lora_a
    expected_weight = torch.stack([
        magnitude * row / row.norm()
        for row, magnitude in zip(direction, adapter.magnitude[:, 0])
    ])
    torch.testing.assert_close(adapter.effective_weight(), expected_weight)
    torch.testing.assert_close(adapter.to_linear()(inputs), adapter(inputs))

    zero_layer = nn.Linear(5, 3, bias=False)
    zero_layer.weight[0].zero_()
    zero_adapter = DoRALinear(zero_layer, rank=2)
    torch.testing.assert_close(zero_adapter(inputs), zero_layer(inputs))
    torch.testing.assert_close(zero_adapter.to_linear()(inputs), zero_adapter(inputs))

print("Проверки инициализации, формулы, градиентов и объединения весов прошли.")

Проверки инициализации, формулы, градиентов и объединения весов прошли.


## Предобучение и адаптация на CPU

Создадим две задачи регрессии с общим скрытым представлением. Сначала модель учится решать исходную задачу целиком. Для целевой задачи отдельно сдвинем направления строк выходного веса низкоранговым обновлением и изменим их длины. Такая постановка позволяет проверить обе части DoRA в контролируемых условиях.

Для адаптации заморозим имеющуюся сеть и заменим только последний `nn.Linear`. Метрику для целевой задачи считаем на отдельной выборке, которая не участвует в оптимизации.

In [11]:
torch.manual_seed(42)
previous_num_threads = torch.get_num_threads()
torch.set_num_threads(1)

input_size = 16
hidden_size = 32
output_size = 8
rank = 2

teacher_encoder = nn.Linear(input_size, hidden_size)
teacher_head = nn.Linear(hidden_size, output_size)

source_inputs = torch.randn(2048, input_size)
target_train_inputs = torch.randn(1024, input_size)
target_test_inputs = torch.randn(512, input_size)

with torch.no_grad():
    source_features = torch.tanh(teacher_encoder(source_inputs))
    source_labels = teacher_head(source_features)

model = nn.Sequential(
    nn.Linear(input_size, hidden_size),
    nn.Tanh(),
    nn.Linear(hidden_size, output_size),
)

pretrain_optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
for _ in range(300):
    pretrain_optimizer.zero_grad()
    loss = F.mse_loss(model(source_inputs), source_labels)
    loss.backward()
    pretrain_optimizer.step()

pretrain_optimizer.zero_grad(set_to_none=True)
with torch.no_grad():
    source_mse = F.mse_loss(model(source_inputs), source_labels).item()
print(f"MSE исходной задачи после предобучения: {source_mse:.6f}")

MSE исходной задачи после предобучения: 0.001127


In [12]:
# Строим целевую задачу с тем же энкодером и выходным слоем с другими весами.
with torch.no_grad():
    source_weight = model[2].weight
    source_bias = model[2].bias
    delta_a = torch.randn(rank, hidden_size) / math.sqrt(hidden_size)
    delta_b = 0.3 * torch.randn(output_size, rank)
    target_direction = F.normalize(source_weight + delta_b @ delta_a, dim=1)
    magnitude_change = torch.linspace(0.5, 1.8, output_size).unsqueeze(1)
    target_magnitude = source_weight.norm(dim=1, keepdim=True) * magnitude_change
    target_weight = target_direction * target_magnitude

    train_features = model[1](model[0](target_train_inputs))
    test_features = model[1](model[0](target_test_inputs))
    target_train_labels = F.linear(train_features, target_weight, source_bias)
    target_test_labels = F.linear(test_features, target_weight, source_bias)

    predictions_before = model(target_test_inputs)
    mse_before = F.mse_loss(predictions_before, target_test_labels).item()

model.requires_grad_(False)
model[2] = DoRALinear(model[2], rank=rank)

with torch.no_grad():
    torch.testing.assert_close(model(target_test_inputs), predictions_before)

trainable_parameters = [param for param in model.parameters() if param.requires_grad]
parameter_count = sum(param.numel() for param in trainable_parameters)
expected_count = rank * (hidden_size + output_size) + output_size
assert parameter_count == expected_count

optimizer = torch.optim.Adam(trainable_parameters, lr=0.02)
for step in range(1, 301):
    optimizer.zero_grad()
    predictions = model(target_train_inputs)
    loss = F.mse_loss(predictions, target_train_labels)
    loss.backward()
    optimizer.step()

    if step in (1, 50, 150, 300):
        print(f"Шаг {step:3d} | train MSE: {loss.item():.6f}")

model.eval()
with torch.no_grad():
    predictions_after = model(target_test_inputs)
    mse_after = F.mse_loss(predictions_after, target_test_labels).item()

assert mse_after < 0.05 * mse_before, "На синтетической задаче ожидается снижение MSE."
print(f"Обучаемые параметры: {parameter_count}")
print(f"Test MSE до адаптации:    {mse_before:.6f}")
print(f"Test MSE после адаптации: {mse_after:.6f}")

Шаг   1 | train MSE: 0.033100
Шаг  50 | train MSE: 0.000940
Шаг 150 | train MSE: 0.000053
Шаг 300 | train MSE: 0.000007
Обучаемые параметры: 88
Test MSE до адаптации:    0.033783
Test MSE после адаптации: 0.000007


## Объединение весов для инференса

После обучения вычисляем итоговый вес один раз. `to_linear()` возвращает готовый слой с тем же смещением. В прямом проходе объединённой модели остаётся обычный `nn.Linear`. Если обучение нужно продолжить, сохраните исходную модель с адаптером до замены слоя.

In [13]:
model[2] = model[2].to_linear()

with torch.inference_mode():
    merged_predictions = model(target_test_inputs)

torch.testing.assert_close(merged_predictions, predictions_after)
max_error = (merged_predictions - predictions_after).abs().max().item()
print(f"Максимальная разница после объединения: {max_error:.2e}")
torch.set_num_threads(previous_num_threads)

Максимальная разница после объединения: 0.00e+00


## Дообучение DistilBERT на AG News

В AG News четыре класса англоязычных новостей. Имена классов берём из метаданных датасета. Из исходной обучающей выборки выделяем валидацию с сохранением долей классов. Официальную тестовую выборку оцениваем после обучения.

Перед первым запуском установите дополнительные зависимости в ядро ноутбука. Если Jupyter попросит перезапустить ядро, сделайте это перед запуском ячеек.

```python
%pip install "transformers>=4.57,<6" "datasets>=3,<5"
```

Для загрузки данных и весов нужен доступ к Hugging Face. По умолчанию запускается обучение на всей выделенной обучающей выборке. Для пробного прогона на CPU установите `QUICK_RUN = True`. В этом режиме используются случайные подвыборки, включая подвыборку официального теста.

[Датасет AG News](https://huggingface.co/datasets/fancyzhx/ag_news) · [Базовая модель DistilBERT](https://huggingface.co/distilbert/distilbert-base-uncased)

In [14]:
from pathlib import Path

from datasets import load_dataset
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, DataCollatorWithPadding
from transformers import DistilBertForSequenceClassification

DATASET_ID = "fancyzhx/ag_news"
MODEL_ID = "distilbert/distilbert-base-uncased"
OUTPUT_DIR = Path("artifacts/ag_news_dora")
QUICK_RUN = False

SEED = 42
RANK = 8
MAX_LENGTH = 128
BATCH_SIZE = 16
LEARNING_RATE = 3e-4
EPOCHS = 1 if QUICK_RUN else 2

torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Устройство: {device}")

/home/stas/up-transformer/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Устройство: cuda


### Данные и батчи

Токенизатор обрезает тексты до `MAX_LENGTH`. При сборке батча `DataCollatorWithPadding` дополняет последовательности до длины самого длинного текста в этом батче и преобразует поле `label` в `labels`, которое принимает модель.

In [15]:
news = load_dataset(DATASET_ID)
class_names = news["train"].features["label"].names

train_val = news["train"].train_test_split(
    test_size=2000,
    seed=SEED,
    stratify_by_column="label",
)
train_news = train_val["train"]
val_news = train_val["test"]
test_news = news["test"]

if QUICK_RUN:
    train_news = train_news.shuffle(seed=SEED).select(range(512))
    val_news = val_news.shuffle(seed=SEED).select(range(128))
    test_news = test_news.shuffle(seed=SEED).select(range(128))

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
collate_batch = DataCollatorWithPadding(tokenizer=tokenizer)


def tokenize_news(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        return_token_type_ids=False,
    )


def make_news_loader(split, *, shuffle=False):
    encoded = split.map(tokenize_news, batched=True, remove_columns=["text"])
    return DataLoader(
        encoded,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        collate_fn=collate_batch,
    )


train_loader = make_news_loader(train_news, shuffle=True)
val_loader = make_news_loader(val_news)
test_loader = make_news_loader(test_news)
print(f"Примеры: train={len(train_news):,}, val={len(val_news):,}, test={len(test_news):,}")
print(f"Классы: {class_names}")

Map: 100%|██████████| 7600/7600 [00:00<00:00, 25289.70 examples/s]

Примеры: train=118,000, val=2,000, test=7,600
Классы: ['World', 'Sports', 'Business', 'Sci/Tech']


### Подключение DoRA к attention

Используем предобученный энкодер DistilBERT. В каждом блоке attention заменяем проекции запросов и значений (`q_lin` и `v_lin`) на уже реализованный `DoRALinear`. Параметры энкодера замораживаем. На новой задаче обучаются матрицы `A` и `B`, векторы `magnitude` и два слоя классификационной головы. Предупреждение при загрузке о случайной инициализации головы ожидаемо.

Проверяем на первом валидационном батче, что замена проекций сама по себе сохраняет логиты модели.

In [16]:
news_model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_ID,
    num_labels=len(class_names),
    id2label=dict(enumerate(class_names)),
    label2id={name: index for index, name in enumerate(class_names)},
    dtype=torch.float32,
).to(device).eval()
base_parameter_count = sum(param.numel() for param in news_model.parameters())


def batch_to_device(batch):
    return {name: tensor.to(device) for name, tensor in batch.items()}


probe_inputs = batch_to_device(next(iter(val_loader)))
probe_inputs.pop("labels")
with torch.no_grad():
    logits_before_dora = news_model(**probe_inputs).logits

news_model.requires_grad_(False)
for block in news_model.distilbert.transformer.layer:
    attention = block.attention
    attention.q_lin = DoRALinear(attention.q_lin, rank=RANK)
    attention.v_lin = DoRALinear(attention.v_lin, rank=RANK)

news_model.pre_classifier.requires_grad_(True)
news_model.classifier.requires_grad_(True)
news_model.eval()

with torch.no_grad():
    torch.testing.assert_close(news_model(**probe_inputs).logits, logits_before_dora)

trainable_parameters = [param for param in news_model.parameters() if param.requires_grad]
trainable_count = sum(param.numel() for param in trainable_parameters)
print(
    f"Обучаемые параметры: {trainable_count:,} "
    f"({trainable_count / base_parameter_count:.2%} от числа параметров исходной модели)"
)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 2577.78it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Обучаемые параметры: 750,340 (1.12% от числа параметров исходной модели)


### Обучение и оценка

Для оптимизации используем среднюю кросс-энтропию. В конце каждой эпохи считаем loss и accuracy на валидации. Оба показателя усредняются по примерам, поэтому короткий последний батч получает соответствующий вес.

In [17]:
@torch.inference_mode()
def evaluate_news(model, loader):
    model.eval()
    loss_sum = 0.0
    correct_count = 0
    example_count = 0

    for batch in loader:
        batch = batch_to_device(batch)
        labels = batch["labels"]
        output = model(**batch)

        batch_size = labels.numel()
        loss_sum += output.loss.item() * batch_size
        correct_count += (output.logits.argmax(dim=-1) == labels).sum().item()
        example_count += batch_size

    return loss_sum / example_count, correct_count / example_count


news_optimizer = torch.optim.AdamW(trainable_parameters, lr=LEARNING_RATE)

for epoch in range(1, EPOCHS + 1):
    news_model.train()
    train_loss_sum = 0.0

    for step, batch in enumerate(train_loader, start=1):
        batch = batch_to_device(batch)
        news_optimizer.zero_grad(set_to_none=True)
        loss = news_model(**batch).loss
        loss.backward()
        news_optimizer.step()

        train_loss_sum += loss.item() * batch["labels"].numel()
        if step % 250 == 0:
            print(f"Эпоха {epoch} | батч {step}/{len(train_loader)} | loss={loss.item():.4f}")

    train_loss = train_loss_sum / len(train_loader.dataset)
    val_loss, val_accuracy = evaluate_news(news_model, val_loader)
    print(
        f"Эпоха {epoch}/{EPOCHS} | train loss={train_loss:.4f} | "
        f"val loss={val_loss:.4f} | val accuracy={val_accuracy:.2%}"
    )

test_loss, test_accuracy = evaluate_news(news_model, test_loader)
print(f"Test ({len(test_news):,} примеров) | loss={test_loss:.4f} | accuracy={test_accuracy:.2%}")

Эпоха 1 | батч 250/7375 | loss=0.5120
Эпоха 1 | батч 500/7375 | loss=0.3134
Эпоха 1 | батч 750/7375 | loss=0.6454
Эпоха 1 | батч 1000/7375 | loss=0.2743
Эпоха 1 | батч 1250/7375 | loss=0.2738
Эпоха 1 | батч 1500/7375 | loss=0.1352
Эпоха 1 | батч 1750/7375 | loss=0.1641
Эпоха 1 | батч 2000/7375 | loss=0.0673
Эпоха 1 | батч 2250/7375 | loss=0.5905
Эпоха 1 | батч 2500/7375 | loss=0.0268
Эпоха 1 | батч 2750/7375 | loss=0.4070
Эпоха 1 | батч 3000/7375 | loss=0.1981
Эпоха 1 | батч 3250/7375 | loss=0.2689
Эпоха 1 | батч 3500/7375 | loss=0.1235
Эпоха 1 | батч 3750/7375 | loss=0.1437
Эпоха 1 | батч 4000/7375 | loss=0.4543
Эпоха 1 | батч 4250/7375 | loss=0.2236
Эпоха 1 | батч 4500/7375 | loss=0.0488
Эпоха 1 | батч 4750/7375 | loss=0.1341
Эпоха 1 | батч 5000/7375 | loss=0.4005
Эпоха 1 | батч 5250/7375 | loss=0.1193
Эпоха 1 | батч 5500/7375 | loss=0.0522
Эпоха 1 | батч 5750/7375 | loss=0.5310
Эпоха 1 | батч 6000/7375 | loss=0.0432
Эпоха 1 | батч 6250/7375 | loss=0.1025
Эпоха 1 | батч 6500/7375 | l

### Экспорт дообученной модели

Объединяем адаптеры с исходными весами и проверяем логиты на том же валидационном батче. После полного прогона сохраняем обычную модель Transformers и токенизатор в `artifacts/ag_news_dora`. В режиме `QUICK_RUN` объединение проверяется, а запись на диск пропускается.

Сохранённую модель можно загрузить через `DistilBertForSequenceClassification.from_pretrained(OUTPUT_DIR)`, а токенизатор через `AutoTokenizer.from_pretrained(OUTPUT_DIR)`.

In [18]:
news_model.eval()
with torch.no_grad():
    logits_before_merge = news_model(**probe_inputs).logits

for block in news_model.distilbert.transformer.layer:
    attention = block.attention
    attention.q_lin = attention.q_lin.to_linear()
    attention.v_lin = attention.v_lin.to_linear()

with torch.no_grad():
    torch.testing.assert_close(news_model(**probe_inputs).logits, logits_before_merge)

print("Логиты после объединения весов совпадают в пределах численной точности.")
if not QUICK_RUN:
    news_model.save_pretrained(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)
    print(f"Модель и токенизатор сохранены в {OUTPUT_DIR}")

Логиты после объединения весов совпадают в пределах численной точности.


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.87it/s]

Модель и токенизатор сохранены в artifacts/ag_news_dora
